### **Check version Python/TPU/GPU**

In [ ]:

import sys, platform, torch, os, subprocess, textwrap
print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())
print("Torch:", torch.__version__)
!nvidia-smi


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
CUDA available: True
Torch: 2.9.0+cu126
Thu Dec 11 04:29:35 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             11W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                        

In [ ]:
# Dowload ultralutics
%pip install ultralytics

####**Yolov8s**

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data='path_file_yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer= 'auto',
    mask_ratio=4,
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0,
    close_mosaic=10,
    name='ppe_yolov8s_v1'
)

####**Yolov11s**

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov11s.pt')

results = model.train(
    data='path_file_yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer= 'auto',
    mask_ratio=4,
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0,
    close_mosaic=10,
    name='ppe_yolov11s_v1'
)

####**Yolov11s_hybrid**

This module contains custom attention mechanisms for YOLOv11 architecture enhancements.

## Overview

Custom PyTorch modules designed to improve feature representation in object detection tasks by incorporating attention mechanisms.

## Components

### CBAM (Convolutional Block Attention Module)
A lightweight attention module that sequentially applies channel and spatial attention to refine feature maps.

**Key Features:**
- **Channel Attention**: Focuses on "what" is meaningful (which feature channels are important)
- **Spatial Attention**: Focuses on "where" is meaningful (which spatial locations are important)
- Minimal computational overhead
- Easy integration with existing CNN architectures

In [ ]:
%%writefile yolo_custom_modules.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics.nn.modules.conv import Conv


class ChannelAttention(nn.Module):
    """Channel Attention Module"""
    def __init__(self, in_channels, reduction=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)


class SpatialAttention(nn.Module):
    """Spatial Attention Module"""
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        padding = (kernel_size - 1) // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        out = self.conv(out)
        return self.sigmoid(out)


class CBAM(nn.Module):
    """
    CBAM: Convolutional Block Attention Module

    Paper: "CBAM: Convolutional Block Attention Module"

    Args:
        c1 (int): Input channels
        c2 (int): Output channels (if None, same as c1)
        reduction (int): Channel reduction ratio for channel attention
        kernel_size (int): Kernel size for spatial attention

    Example:
        >>> import torch
        >>> x = torch.randn(1, 256, 40, 40)
        >>> cbam = CBAM(256)
        >>> out = cbam(x)
        >>> print(out.shape)
        torch.Size([1, 256, 40, 40])
    """
    def __init__(self, c1, c2=None, reduction=16, kernel_size=7):
        super(CBAM, self).__init__()
        c2 = c2 or c1

        self.channel_attention = ChannelAttention(c1, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)

        # If input and output channels differ, add 1x1 conv
        self.match_channels = nn.Identity() if c1 == c2 else nn.Conv2d(c1, c2, 1)

    def forward(self, x):
        # Channel attention
        x = x * self.channel_attention(x)

        # Spatial attention
        x = x * self.spatial_attention(x)

        # Match output channels if needed
        x = self.match_channels(x)

        return x
class SPDConv(nn.Module):
    """
    SPDConv: Space-to-Depth + Conv
    Replacement for Conv(stride=2) but keeps more spatial info.
    """
    def __init__(self, c1, c2, k=3):
        super().__init__()
        self.k = k
        self.conv = nn.Conv2d(c1 * 4, c2, k, stride=1, padding=k // 2, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU()

    def space_to_depth(self, x):
        # B, C, H, W → B, 4C, H/2, W/2
        return torch.cat([
            x[..., ::2, ::2],       # top-left
            x[..., 1::2, ::2],      # bottom-left
            x[..., ::2, 1::2],      # top-right
            x[..., 1::2, 1::2],     # bottom-right
        ], dim=1)

    def forward(self, x):
        x = self.space_to_depth(x)
        x = self.conv(x)
        x = self.bn(x)
        return self.act(x)




Writing yolo_custom_modules.py


In [ ]:
%%writefile yolo11s_hybrid.yaml
nc: 9  # number of classes (9 PPE classes)
scales:
  s: [0.50, 0.50, 1024]
# YOLOv11s-PPE backbone (ORIGINAL - NO CHANGES)
backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]]  # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]  # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]  # 2
  - [-1, 1, Conv, [256, 3, 2]]  # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]  # 4
  - [-1, 1, Conv, [512, 3, 2]]  # 5-P4/16
  - [-1, 2, C3k2, [512, True]]  # 6
  - [-1, 1, Conv, [1024, 3, 2]]  # 7-P5/32
  - [-1, 2, C3k2, [1024, True]]  # 8
  - [-1, 1, SPPF, [1024, 5]]  # 9
  - [-1, 2, C2PSA, [1024]]  # 10

# YOLOv11s-PPE head with 3 KEY CHANGES
head:
  # P4 upsampling
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]  # 11
  - [[-1, 6], 1, Concat, [1]]  # 12 cat backbone P4
  - [-1, 2, C3k2, [512, False]]  # 13

  # P3 upsampling + CHANGE 2: Add CBAM here (small object focus)
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]  # 14
  - [[-1, 4], 1, Concat, [1]]  # 15 cat backbone P3
  - [-1, 2, C3k2, [256, False]]  # 16
  - [-1, 1, CBAM, [128]]  # 17 - CBAM attention for small objects

  # CHANGE 1: Add P2 detection head for very small objects (gloves, shoes)
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]  # 18
  - [[-1, 2], 1, Concat, [1]]  # 19 cat backbone P2
  - [-1, 2, C3k2, [128, False]]  # 20 - P2 features

  # P3 downsample
  - [20, 1, Conv, [256, 3, 2]]  # 21
  - [[-1, 17], 1, Concat, [1]]  # 22 cat head P3
  - [-1, 2, C3k2, [256, False]]  # 23

  # P4 downsample
  - [-1, 1, Conv, [512, 3, 2]]  # 24
  - [[-1, 13], 1, Concat, [1]]  # 25 cat head P4
  - [-1, 2, C3k2, [512, True]]  # 26

  # P5 downsample
  - [-1, 1, Conv, [512, 3, 2]]  # 27
  - [[-1, 10], 1, Concat, [1]]  # 28 cat head P5
  - [-1, 2, C3k2, [1024, True]]  # 29

  # CHANGE 1: Detect with 4 heads [P2, P3, P4, P5] instead of 3
  - [[20, 23, 26, 29], 1, Detect, [nc]]  # 30 Detect(P2, P3, P4, P5)


Writing yolo11s_hybrid.yaml


In [ ]:
# run_yolo_custom_safe.py
import os
import sys
import traceback
import importlib.util

MODULE_PATH = "yolo_custom_modules.py"  # path yolov11s_hybrid.py
YAML_PATH = "yolo11s_hybrid.yaml"       # path yaml yolov11s_hybrid.yaml

def load_module_from_path(module_name, path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Module file not found: {path}")
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    try:
        spec.loader.exec_module(module)
    except Exception as e:
        print(f"Error while importing module {path}: {e}")
        traceback.print_exc()
        raise
    return module

def main():
    print("Python sys.path:")
    for p in sys.path:
        print(" ", p)
    print("Checking module file:", MODULE_PATH)
    if not os.path.exists(MODULE_PATH):
        print("-> The file does not exist. Check the MODULE_PATH.")
        return

    # Ensure ultralytics is importable (so that Conv import inside module works)
    try:
        import ultralytics
        print("Ultralytics version:", getattr(ultralytics, '__version__', 'unknown'))
    except Exception as e:
        print("Unable to import 'ultralytics'. Install/check the version before continuing..")
        traceback.print_exc()
        return

    # Load the custom module by explicit path
    try:
        custom = load_module_from_path("yolo_custom_modules", MODULE_PATH)
        print("Custom module has been successfully loaded.")
    except Exception as e:
        print("Unable to load custom module. Stop.")
        return

    # Extract classes and register into ultralytics.nn.tasks
    missing = []
    for name in ("SPDConv", "CBAM"):
        if not hasattr(custom, name):
            missing.append(name)
    if missing:
        print("The module is loaded but the class is missing:", missing)
        print("List of attributes in the module:", dir(custom))
        return


    CBAM = getattr(custom, "CBAM")
    SPDConv = getattr(custom, "SPDConv")
    # Register into module ultralytics.nn.tasks
    try:
        import ultralytics.nn.tasks as tasks_mod
        # Assign directly to the module namespace.

        setattr(tasks_mod, "CBAM", CBAM)
        setattr(tasks_mod, "SPDConv", SPDConv)

        tasks_mod.__dict__["CBAM"] = CBAM
        tasks_mod.__dict__["SPDConv"] = SPDConv
        print("Have registered AttnBlock and CBAM with ultralytics.nn.tasks.")
    except Exception as e:
        print("Unable to register at ultralytics.nn.tasks:")
        traceback.print_exc()
        return

    print("tasks_mod.CBAM:", tasks_mod.__dict__.get("CBAM"))
    print("tasks_mod.SPDConv:", tasks_mod.__dict__.get("SPDConv"))

    try:
        from ultralytics import YOLO
        model = YOLO(YAML_PATH)
        print("YOLO model created successfully from", YAML_PATH)
    except Exception as e:
        print("Lỗi khi tạo YOLO model (sau khi đăng ký lớp):")
        traceback.print_exc()
        return

if __name__ == "__main__":
    main()


In [ ]:
import sys
import os

# if '/content' not in sys.path:
#     sys.path.insert(0, '/content')

import yolo_custom_modules

import ultralytics.nn.tasks as tasks
tasks.CBAM = yolo_custom_modules.CBAM

from ultralytics import YOLO

if __name__ == "__main__":
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:64"

    model = YOLO('yolo11s_hybrid.yaml').load('yolo11s.pt')

    model.train(
        data='path_file_yaml',
        epochs=100,
        imgsz=640,
        batch=16,
        optimizer= 'auto',
        mask_ratio=4,
        lr0=0.01,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3.0,
        warmup_momentum=0.8,
        warmup_bias_lr=0.1,
        box=7.5,
        cls=0.5,
        dfl=1.5,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        flipud=0.0,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.0,
        close_mosaic=10,
        name='ppe_yolov11s_hybrid'
    )

###Demo detect images by model

In [ ]:
from ultralytics import YOLO
model = YOLO('path_best.pt')

model.predict(
    source='source_image',
    save=True,
    imgsz=640
)
